# Día 13 · Automatización de extremo a extremo

Selecciona **Run all**. El notebook te pedirá cargar el ZIP o CSV privado y luego ingresar la clave OpenAI en un campo protegido. El resto se ejecuta automáticamente. Los archivos privados y checkpoints no se suben a GitHub.

In [ ]:
!if [ ! -d /content/proyecto_ActividadGrado-Riesgos ]; then git clone -q --branch dia-13-automatizacion-end-to-end --single-branch https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git /content/proyecto_ActividadGrado-Riesgos; fi
%cd /content/proyecto_ActividadGrado-Riesgos
!pip -q install -r requirements.txt

In [ ]:
import os
import zipfile
from getpass import getpass
from pathlib import Path
from google.colab import files

ROOT = Path('/content/proyecto_ActividadGrado-Riesgos')
PRIVATE_DIR = Path('/content/private_day_13')
PRIVATE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PRIVATE_DIR)
print('Carga resultados_clasificador_calibrado_dia_05.zip o documentary_items_calibrated.csv')
uploaded = files.upload()
for filename in uploaded:
    path = PRIVATE_DIR / filename
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as archive:
            archive.extractall(PRIVATE_DIR / 'extraido')
catalogs = list(PRIVATE_DIR.rglob('documentary_items_calibrated.csv'))
if not catalogs:
    raise FileNotFoundError('El archivo cargado no contiene documentary_items_calibrated.csv')
PRIVATE_CATALOG = catalogs[0]
PRIVATE_CHECKPOINTS = PRIVATE_DIR / 'checkpoints_day_13'
os.chdir(ROOT)

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

EMBEDDINGS = ROOT / 'data/processed/embedding/embeddings_bge_m3.parquet'
assert EMBEDDINGS.exists(), EMBEDDINGS
print('Catálogo privado encontrado:', PRIVATE_CATALOG.name)

In [ ]:
from src.pipeline.end_to_end import (
    BgeM3Retriever, CalibratedCatalogExtractor, EndToEndPipeline,
    JsonCheckpointStore, OpenAIRagAnswerer, PublishedProfileLoader,
    public_execution_summary,
)

pipeline = EndToEndPipeline(
    retriever=BgeM3Retriever(EMBEDDINGS, top_k=5),
    extractor=CalibratedCatalogExtractor(PRIVATE_CATALOG),
    profiler=PublishedProfileLoader(
        ROOT / 'results/day_11/intelligent_risk_profile.json',
        ROOT / 'results/day_11/category_profile.csv',
    ),
    answerer=OpenAIRagAnswerer(model='gpt-4o-mini'),
    checkpoint_store=JsonCheckpointStore(PRIVATE_CHECKPOINTS),
)

In [ ]:
QUESTION = '¿Qué retrasos o incumplimientos requieren vigilancia y qué evidencia los sustenta?'
result = pipeline.run(QUESTION, request_id='DEMO-DIA-13', resume=True)
public_execution_summary(result)

## Verificación privada

Revisa `result['qa_result']` y `result['risk_result']` únicamente dentro de Colab. No descargues ni publiques el checkpoint completo. El objeto mostrado arriba es el resumen seguro para documentación.